In [1]:
# Cell 1: Dependencies & Repository Setup
!git clone https://github.com/Project-DiffShield/DiffShield.git
import sys
sys.path.append('/kaggle/working/DiffShield')

!pip install -q kornia diffusers transformers optuna lpips accelerate scikit-learn kneed
print("Environment dependencies initialized.")

Cloning into 'DiffShield'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 21 (delta 3), reused 21 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 424.00 KiB | 32.62 MiB/s, done.
Resolving deltas: 100% (3/3), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.7 MB/s eta 0:00:00
Environment dependencies initialized.


In [2]:
# Cell 2: Models & Perceptual Metric Evaluation
import torch
import numpy as np
import math
import os
import shutil
import zipfile
import torchvision.transforms as T
from PIL import Image
import lpips
from torchvision.utils import save_image

from src.losses import DiffShieldLoss
from src.optimizer import PGDOptimizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Executing on device: {device}")

# Core evaluation models
loss_fn = DiffShieldLoss(device=device)
lpips_vgg = lpips.LPIPS(net='vgg').to(device)
target_concept_embedding = loss_fn.encode_target_text(["a potted plant"])

def compute_image_quality_metrics(clean_tensor, immunized_tensor):
    """
    Computes strict visual and deep perceptual divergence metrics:
    - PSNR: Peak Signal-to-Noise Ratio (dB)
    - SSIM: Structural Similarity Index
    - LPIPS: VGG-based Perceptual Feature Distance
    - Linf: Maximum allowable single-pixel perturbation delta
    """
    clean_np = ((clean_tensor.squeeze(0).cpu().numpy() + 1.0) * 127.5).astype(np.uint8)
    immunized_np = ((immunized_tensor.squeeze(0).cpu().numpy() + 1.0) * 127.5).astype(np.uint8)
    
    mse = np.mean((clean_np.astype(np.float64) - immunized_np.astype(np.float64)) ** 2)
    psnr = 20 * math.log10(255.0 / math.sqrt(mse)) if mse > 0 else float('inf')
    
    C1 = (0.01 * 255) ** 2
    C2 = (0.03 * 255) ** 2
    mu1, mu2 = clean_np.mean(), immunized_np.mean()
    s1_sq, s2_sq = clean_np.var(), immunized_np.var()
    s12 = ((clean_np - mu1) * (immunized_np - mu2)).mean()
    ssim = ((2 * mu1 * mu2 + C1) * (2 * s12 + C2)) / ((mu1 ** 2 + mu2 ** 2 + C1) * (s1_sq + s2_sq + C2))
    
    with torch.no_grad():
        lpips_score = lpips_vgg(clean_tensor, immunized_tensor).item()
        
    linf = (immunized_tensor - clean_tensor).abs().max().item()
    return {"PSNR": psnr, "SSIM": ssim, "LPIPS": lpips_score, "Linf": linf}

print("Backbones and metric computation functions initialized.")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Executing on device: cuda


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 202MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/vgg.pth
Backbones and metric computation functions initialized.


In [3]:
# Cell 3: Dataset Extraction & Attribute Matrix Loading
import pandas as pd

dataset_dir = '/kaggle/working/celeba_extracted/CelebAHQ/Img/hq/data512x512'
os.makedirs(dataset_dir, exist_ok=True)

# Locate and extract the dataset archive
zip_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.zip') and ('celeba' in f.lower() or 'hq' in f.lower()):
            zip_path = os.path.join(root, f)
            break
    if zip_path:
        break

if zip_path:
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_dir)
else:
    # Look for raw directory if not zipped
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.lower().endswith(('.png', '.jpg', '.jpeg')) for f in files):
            dataset_dir = root
            break

# Locate attribute text file
attr_file_path = None
for root, dirs, files in os.walk('/kaggle'):
    for f in files:
        if 'attr' in f.lower() and f.endswith('.txt'):
            attr_file_path = os.path.join(root, f)
            break
    if attr_file_path:
        break

# Parse attribute file into coordinate matrix
with open(attr_file_path, 'r') as f:
    lines = [line.strip() for line in f.readlines() if line.strip()]

num_images = int(lines[0])
attr_names = lines[1].split()
data = []
img_filenames = []

for line in lines[2:]:
    parts = line.split()
    img_filenames.append(parts[0])
    data.append([1 if int(x) == 1 else 0 for x in parts[1:]])

attr_matrix = np.array(data, dtype=np.float32)
print(f"Loaded {attr_matrix.shape[0]} images across {attr_matrix.shape[1]} binary attribute dimensions.")

TypeError: expected str, bytes or os.PathLike object, not NoneType

In [ ]:
# Cell 4: Elbow Point Detection & Macro-Centroid Extraction
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min
from kneed import KneeLocator

wcss = []
k_range = range(1, 21)

print("Computing WCSS across candidate cluster ranges (1 to 20)...")
for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    km.fit(attr_matrix)
    wcss.append(km.inertia_)

# Determine the exact inflection point
kl = KneeLocator(k_range, wcss, curve="convex", direction="decreasing")
k_calib = kl.elbow if kl.elbow is not None else 8
print(f"\nMathematical Elbow Detected at K = {k_calib}")
print(f"Calibration Subset size locked at {k_calib} representative macro-archetypes.")

# Cluster at k_calib and extract centroids
kmeans_calib = KMeans(n_clusters=k_calib, init='k-means++', random_state=42, n_init=10)
kmeans_calib.fit(attr_matrix)

centroid_indices_calib, _ = pairwise_distances_argmin_min(kmeans_calib.cluster_centers_, attr_matrix)
calib_filenames = [img_filenames[idx] for idx in centroid_indices_calib]

calib_dir = '/kaggle/working/calibration_subset'
os.makedirs(calib_dir, exist_ok=True)

for fname in calib_filenames:
    for root, dirs, files in os.walk(dataset_dir):
        if fname in files:
            shutil.copy(os.path.join(root, fname), os.path.join(calib_dir, fname))
            break

print(f"Copied {len(os.listdir(calib_dir))} macro-centroid images to {calib_dir}")

In [ ]:
# Cell 5: Disjoint 70-Centroid Extraction (Data Isolation)
# Exclude calibration indices to guarantee no overlap
calib_set = set(centroid_indices_calib)
eval_indices_available = [i for i in range(len(img_filenames)) if i not in calib_set]

eval_attr_matrix = attr_matrix[eval_indices_available]
eval_filenames_available = [img_filenames[i] for i in eval_indices_available]

K_EVAL = 70
print(f"Clustering {eval_attr_matrix.shape[0]} disjoint images into {K_EVAL} attribute centroids...")
kmeans_eval = KMeans(n_clusters=K_EVAL, init='k-means++', random_state=42, n_init=10)
kmeans_eval.fit(eval_attr_matrix)

centroid_indices_eval, _ = pairwise_distances_argmin_min(kmeans_eval.cluster_centers_, eval_attr_matrix)
eval_final_filenames = [eval_filenames_available[idx] for idx in centroid_indices_eval]

eval_dir = '/kaggle/working/diverse_70_images'
os.makedirs(eval_dir, exist_ok=True)

for fname in eval_final_filenames:
    for root, dirs, files in os.walk(dataset_dir):
        if fname in files:
            shutil.copy(os.path.join(root, fname), os.path.join(eval_dir, fname))
            break

print(f"Data Isolation Complete: Extracted {len(os.listdir(eval_dir))} disjoint evaluation centroids.")

In [ ]:
# Cell 6: Base Normalization Multiplier Calculation
from torch.utils.data import DataLoader
from src.data import get_dataloader

calib_loader = get_dataloader(root_dir=calib_dir, batch_size=k_calib, image_size=512)
calib_batch, _ = next(iter(calib_loader))
calib_batch = calib_batch.to(device)

# Apply bounded uniform test perturbation (epsilon = 8/255)
epsilon = 8 / 255
delta = torch.zeros_like(calib_batch).to(device)
delta.uniform_(-epsilon, epsilon)
poisoned_batch = torch.clamp(calib_batch + delta, -1.0, 1.0)

# Measure baseline raw multi-objective losses across the batch
raw_vis = loss_fn.compute_visual_loss(calib_batch, poisoned_batch).item()
raw_sem = loss_fn.compute_semantic_loss(poisoned_batch, target_concept_embedding).item()
raw_str = loss_fn.compute_structure_loss(calib_batch, poisoned_batch).item()

print(f"Subset Baseline Raw Losses -> Vis: {raw_vis:.4f} | Sem: {raw_sem:.4f} | Str: {raw_str:.4f}")

# Compute inverse scaling constants
alpha_base = 1.0 / max(raw_vis, 1e-4)
beta_base  = 1.0 / max(raw_sem, 1e-4)
gamma_base = 1.0 / max(raw_str, 1e-4)

print(f"\nCalibrated Base Multipliers:")
print(f"  alpha_base (Visual)     = {alpha_base:.4f}")
print(f"  beta_base  (Semantic)   = {beta_base:.4f}")
print(f"  gamma_base (Structural) = {gamma_base:.4f}")

In [ ]:
# Cell 7: Empirical Bound Sweeping
multipliers = [0.1, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0]

def sweep_subset_bounds(param_name):
    valid_mults = []
    print(f"\n--- Sweeping Search Boundaries for {param_name} across Calibration Set ---")
    
    for mult in multipliers:
        w_a = alpha_base * (mult if param_name == 'alpha' else 1.0)
        w_b = beta_base  * (mult if param_name == 'beta'  else 1.0)
        w_g = gamma_base * (mult if param_name == 'gamma' else 1.0)
        
        optimizer = PGDOptimizer(epsilon=8/255, alpha=1/255, iters=40, device=device)
        batch_passed = True
        
        for i in range(calib_batch.size(0)):
            img = calib_batch[i:i+1]
            immunized = optimizer.optimize(img, target_concept_embedding, w_alpha=w_a, w_beta=w_b, w_gamma=w_g)
            m = compute_image_quality_metrics(img, immunized)
            if m['PSNR'] < 38.0 or m['SSIM'] < 0.95:
                batch_passed = False
                break
                
        status = "PASS" if batch_passed else "FAIL"
        print(f"Multiplier {mult:4.2f}x -> Constraint Status: {status}")
        if batch_passed:
            valid_mults.append(mult)
            
    min_m = min(valid_mults) if valid_mults else 0.1
    max_m = max(valid_mults) if valid_mults else 1.0
    return min_m, max_m

min_a, max_a = sweep_subset_bounds('alpha')
min_b, max_b = sweep_subset_bounds('beta')
min_g, max_g = sweep_subset_bounds('gamma')

min_alpha, max_alpha = alpha_base * min_a, alpha_base * max_a
min_beta,  max_beta  = beta_base  * min_b, beta_base  * max_b
min_gamma, max_gamma = gamma_base * min_g, gamma_base * max_g

print(f"\n==========================================")
print(f"  VALID SEARCH BOUNDARIES FOR OPTUNA")
print(f"==========================================")
print(f"  alpha : [{min_alpha:.4f}, {max_alpha:.4f}]")
print(f"  beta  : [{min_beta:.4f}, {max_beta:.4f}]")
print(f"  gamma : [{min_gamma:.4f}, {max_gamma:.4f}]")

In [ ]:
# Cell 8: Bayesian Optimization Across Calibration Batch
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    w_a = trial.suggest_float('alpha', min_alpha, max_alpha)
    w_b = trial.suggest_float('beta',  min_beta,  max_beta)
    w_g = trial.suggest_float('gamma', min_gamma, max_gamma)
    
    optimizer = PGDOptimizer(epsilon=8/255, alpha=1/255, iters=40, device=device)
    subset_losses = []
    
    for i in range(calib_batch.size(0)):
        img = calib_batch[i:i+1]
        immunized = optimizer.optimize(img, target_concept_embedding, w_alpha=w_a, w_beta=w_b, w_gamma=w_g)
        m = compute_image_quality_metrics(img, immunized)
        
        # Universal constraint enforcement
        if m['PSNR'] < 38.0 or m['SSIM'] < 0.95:
            return -9999.0
            
        total_loss, _, _, _ = loss_fn(img, immunized, target_concept_embedding, alpha=w_a, beta=w_b, gamma=w_g)
        subset_losses.append(total_loss.item())
        
    return sum(subset_losses) / len(subset_losses)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

opt_alpha = study.best_params['alpha']
opt_beta  = study.best_params['beta']
opt_gamma = study.best_params['gamma']

print("\n==================================================")
print("  FINAL OPTIMAL GENERALIZED HYPERPARAMETERS")
print("==================================================")
print(f"  Optimal Alpha (Visual)     = {opt_alpha:.4f}")
print(f"  Optimal Beta  (Semantic)   = {opt_beta:.4f}")
print(f"  Optimal Gamma (Structural) = {opt_gamma:.4f}")
print(f"  Best Mean Calibration Loss = {study.best_value:.4f}")

In [ ]:
# Cell 9: Execution Across 70 Evaluation Centroids
eval_loader = get_dataloader(root_dir=eval_dir, batch_size=1, image_size=512)
output_protected_dir = '/kaggle/working/stage1_protected_images'
os.makedirs(output_protected_dir, exist_ok=True)

results_records = []
optimizer = PGDOptimizer(epsilon=8/255, alpha=1/255, iters=40, device=device)

print(f"Processing 70 Disjoint Centroids using Optimal Hyperparameters...")
for idx, (clean_img, path) in enumerate(eval_loader):
    clean_img = clean_img.to(device)
    
    immunized_img = optimizer.optimize(
        clean_img, target_concept_embedding,
        w_alpha=opt_alpha, w_beta=opt_beta, w_gamma=opt_gamma
    )
    
    metrics = compute_image_quality_metrics(clean_img, immunized_img)
    results_records.append(metrics)
    
    # Save protected image to disk in standard [0, 1] range
    protected_norm = (immunized_img.squeeze(0) + 1.0) / 2.0
    filename = f"protected_face_{idx+1:03d}.png"
    save_image(protected_norm, os.path.join(output_protected_dir, filename))
    
    if (idx + 1) % 10 == 0 or (idx + 1) == 70:
        print(f"Completed [{idx+1:02d}/70] | PSNR: {metrics['PSNR']:.2f} dB | SSIM: {metrics['SSIM']:.4f} | LPIPS: {metrics['LPIPS']:.4f}")

In [ ]:
# Cell 10: Metrics Table Display & File Compression
psnr_vals = [r['PSNR'] for r in results_records]
ssim_vals = [r['SSIM'] for r in results_records]
lpips_vals = [r['LPIPS'] for r in results_records]

psnr_viols = sum(1 for p in psnr_vals if p < 38.0)
ssim_viols = sum(1 for s in ssim_vals if s < 0.95)
lpips_viols = sum(1 for l in lpips_vals if l > 0.05)

print("\n" + "=" * 70)
print(f"  STAGE 1 GENERALIZATION METRICS ({len(results_records)} Disjoint Images)")
print("=" * 70)
print(f"{'Metric':<10} | {'Mean ± Std':<18} | {'Min (Worst)':<12} | {'Max (Best)':<12} | {'Violations'}")
print("-" * 70)
print(f"{'PSNR (dB)':<10} | {np.mean(psnr_vals):6.2f} ± {np.std(psnr_vals):5.2f}    | {np.min(psnr_vals):12.2f} | {np.max(psnr_vals):12.2f} | {psnr_viols}/{len(psnr_vals)} (< 38.0)")
print(f"{'SSIM':<10} | {np.mean(ssim_vals):6.4f} ± {np.std(ssim_vals):5.4f}  | {np.min(ssim_vals):12.4f} | {np.max(ssim_vals):12.4f} | {ssim_viols}/{len(ssim_vals)} (< 0.95)")
print(f"{'LPIPS':<10} | {np.mean(lpips_vals):6.4f} ± {np.std(lpips_vals):5.4f}  | {np.min(lpips_vals):12.4f} | {np.max(lpips_vals):12.4f} | {lpips_viols}/{len(lpips_vals)} (> 0.05)")
print("=" * 70)

# Compress output bundles
!zip -r -q /kaggle/working/stage1_protected_images.zip /kaggle/working/stage1_protected_images
!zip -r -q /kaggle/working/stage1_original_images.zip /kaggle/working/diverse_70_images
print("\nAll tasks completed. Both protected and original image archives are ready for download.")